# 07 — Aprendizado por reforço para planejamento de estoque em vigilância epidemiológica

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/07_aprendizado_reforco.ipynb)

**Duração estimada:** 75–90 minutos  
**Pré-requisitos:** Python, arrays e noções de séries temporais.

## Objetivos

- distinguir estado, ação, recompensa e episódio
- treinar Q-learning tabular em nove estados
- comparar a política com três baselines em teste temporal
- discutir escolhas normativas da recompensa

## Fonte e licença

Série semanal real da [API InfoDengue — descrição e acesso](https://info.dengue.mat.br/tutorial_api_python/locale-en); ações, estoques e recompensas são sintéticos.

Consulte os termos do InfoDengue. A simulação não representa tratamento nem decisão clínica.

> **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Preparação do ambiente

> Como obter a série real sem substituir falhas por números artificiais?

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.dengue_api import fetch_infodengue, missing_week_intervals
from src.reinforcement_env import DengueInventoryEnv, evaluate_policy, train_q_learning

FAST_MODE = True
GEOCODE = 3303302  # Niterói
USE_RIO_FALLBACK = False  # mude para True para autorizar 3304557 explicitamente

## Pergunta orientadora

> Uma política aprendida em anos iniciais reduz custos da simulação nos anos finais, comparada a regras simples?

## Obtenção dos dados

> A resposta foi validada, ordenada e datada? Há semanas ausentes?

In [ ]:
dengue = fetch_infodengue(
    geocode=GEOCODE, disease="dengue", start_year=2016, end_year=2025,
    fallback_geocode=3304557 if USE_RIO_FALLBACK else None,
)
target = "casos_est" if "casos_est" in dengue else "casos"
print("Fonte/cache e download:", dengue.attrs)
print("Alvo:", target, "| intervalo:", dengue.data_iniSE.min(), "a", dengue.data_iniSE.max())
display(missing_week_intervals(dengue).head(10))

### Como interpretar

Semanas ausentes são exibidas e não recebem zero. Valores do InfoDengue podem ser revisados. Se a API falhar, a execução para com mensagem acionável; nenhum dado sintético substitui a série.

## Preparação temporal

> Os limites de estado e estoque são calculados somente no treino?

In [ ]:
series = dengue.set_index("data_iniSE")[target].pipe(pd.to_numeric, errors="coerce")
missing_values = int(series.isna().sum())
series = series.dropna().sort_index()
split_date = series.index.max() - pd.DateOffset(years=2)
train_series = series[series.index < split_date]
test_series = series[series.index >= split_date]
print(f"Valores ausentes removidos: {missing_values}")
print("Treino:", train_series.index.min(), "a", train_series.index.max(), len(train_series))
print("Teste:", test_series.index.min(), "a", test_series.index.max(), len(test_series))

In [ ]:
stock_levels = np.quantile(train_series, [0.25, 0.50, 0.75])
train_env = DengueInventoryEnv(
    train_series.to_numpy(), stock_levels=stock_levels,
    shortage_cost=5.0, excess_cost=1.0, operational_cost=0.05,
)
print("Estoques simulados (q25, q50, q75 do treino):", stock_levels.round(1))
print("Limites dos estados de demanda:", train_env.demand_thresholds.round(1))

## Ambiente simulado

Cada estado combina demanda recente baixa/média/alta e tendência caindo/estável/subindo. As ações são estoques baixo/médio/alto. A recompensa é o negativo de custo de falta + excesso + operação; falta custa mais por uma escolha explícita de configuração.

## Experimento: Q-learning

> A recompensa por episódio se estabiliza enquanto epsilon diminui?

In [ ]:
q_table, history = train_q_learning(
    train_env, episodes=500 if FAST_MODE else 1_500,
    alpha=0.1, gamma=0.95,
    epsilon_start=1.0, epsilon_end=0.05,
    random_state=RANDOM_STATE,
)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
history.set_index("episódio")["recompensa"].rolling(30, min_periods=1).mean().plot(ax=axes[0], title="Recompensa média móvel")
history.plot(x="episódio", y="epsilon", ax=axes[1], legend=False, title="Exploração (epsilon)")
plt.tight_layout(); plt.show()

In [ ]:
state_labels = [f"{d}/{t}" for d in ["baixa", "média", "alta"] for t in ["caindo", "estável", "subindo"]]
action_labels = ["estoque baixo", "estoque médio", "estoque alto"]
q_frame = pd.DataFrame(q_table, index=state_labels, columns=action_labels)
sns.heatmap(q_frame, cmap="viridis", annot=True, fmt=".0f")
plt.title("Q-table final (maior valor = ação preferida)"); plt.show()
policy = np.argmax(q_table, axis=1)
display(pd.DataFrame({"estado": state_labels, "ação": np.array(action_labels)[policy]}))

## Avaliação temporal

> A política supera estoque médio, aleatório e regra do último nível?

In [ ]:
test_env = DengueInventoryEnv(
    test_series.to_numpy(), stock_levels=stock_levels,
    shortage_cost=train_env.shortage_cost, excess_cost=train_env.excess_cost,
    operational_cost=train_env.operational_cost,
    demand_thresholds=train_env.demand_thresholds,
    trend_tolerance=train_env.trend_tolerance,
)
rng = np.random.default_rng(RANDOM_STATE)
policies = {
    "Q-learning": policy,
    "sempre médio": np.ones(9, dtype=int),
    "aleatória": lambda state, env: int(rng.integers(3)),
    "regra último nível": lambda state, env: state // 3,
}
comparison = pd.DataFrame({name: evaluate_policy(test_env, rule) for name, rule in policies.items()}).T
display(comparison.round(1))

In [ ]:
comparison[["falta", "excesso"]].plot.bar(figsize=(9, 4), title="Custos físicos simulados no teste")
plt.ylabel("Unidades acumuladas"); plt.xticks(rotation=15); plt.show()
comparison["recompensa"].plot.bar(title="Recompensa acumulada no teste (maior é melhor na simulação)")
plt.ylabel("Recompensa"); plt.xticks(rotation=15); plt.show()

### Como interpretar

Maior recompensa significa somente melhor resultado **sob a função escolhida**. Não significa melhor decisão pública. Os quantis, estados e custos vieram do treino; o teste temporal não atualiza a política. Os dados históricos não mostram o efeito causal das ações simuladas.

## Limitações e responsabilidade

- Estado, ações e recompensa simplificam logística, validade, capacidade, orçamento e incerteza.
- Dar peso maior à falta é uma escolha normativa, não uma propriedade descoberta nos dados.
- A demanda é real, mas nenhuma ação foi aplicada; não há evidência causal de impacto.

## Atividade

Dobre o custo de excesso ou de falta, treine novamente e compare a política. Explique por que otimizar a nova recompensa não resolve sozinho uma decisão pública.

## Três aprendizados principais

1. Q-learning aprende valores de estado-ação por tentativa na simulação.
2. Limites e política devem ser aprendidos antes do teste temporal.
3. A função de recompensa incorpora valores e precisa ser debatida.

## Referências

- [InfoDengue — tutorial da API](https://info.dengue.mat.br/tutorial_api_python/locale-en)
- Sutton & Barto. Reinforcement Learning: An Introduction, 2ª ed.

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'matplotlib', 'seaborn', 'requests'))